># Mutation Signature Feature Engineering
>
>## 분석 목적
>아미노산 치환 정보를 바탕으로 가능한 단일 염기 치환을 추론하여, 환자별 6종 염기 치환 패턴을 피처로 생성한다.
>
>- **입력:** 유전자별 아미노산 변이 문자열
>- **출력:** 6종 치환 개수(log1p), 6종 치환 비율, 전체 치환 개수(log1p) — 총 13개 피처
>- **방법:** NCBI 표준 유전 암호표(Translation Table 1)를 이용한 코돈 기반 치환 추론
>- **주의:** 실제 DNA 염기 치환을 직접 관측한 값이 아닌, 가능한 코돈 변화에 기반한 추정값
>
>**Reference:** [NCBI Genetic Codes](https://www.ncbi.nlm.nih.gov/Taxonomy/Utils/wprintgc.cgi)

# 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path.cwd().resolve().parents[1]

print("Project root:", PROJECT_ROOT)

Project root: /Users/admin/Documents/ah07-cancer-subtype


# 2. 로직 구현

In [2]:
# 표준 유전 암호표 및 염기 치환 매핑 함수 작성

from itertools import product

# NCBI Standard Genetic Code (Translation Table 1)
BASES = "TCAG"
AA = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"

CODON_TO_AA = {
    "".join(codon): AA[i]
    for i, codon in enumerate(product(BASES, repeat=3))
}

# 아미노산별 가능한 코돈
AA_TO_CODONS = {}

for codon, amino_acid in CODON_TO_AA.items():
    AA_TO_CODONS.setdefault(amino_acid, []).append(codon)

# 염기 상보 관계
COMPLEMENT = {
    "A": "T",
    "T": "A",
    "C": "G",
    "G": "C"
}

SIGNATURE_TYPES = [
    "C>A", "C>G", "C>T",
    "T>A", "T>C", "T>G"
]


def normalize_substitution(ref, alt):
    """염기 치환을 피리미딘(C/T) 기준 6종으로 정규화."""
    if ref in "AG":
        ref = COMPLEMENT[ref]
        alt = COMPLEMENT[alt]

    return f"{ref}>{alt}"


def infer_substitution_weights(ref_aa, alt_aa):
    """
    아미노산 치환을 설명할 수 있는 단일 염기 치환을 탐색하고,
    6종 치환 유형별 비율을 반환한다.
    """
    counts = dict.fromkeys(SIGNATURE_TYPES, 0)

    for ref_codon in AA_TO_CODONS.get(ref_aa, []):
        for pos in range(3):
            for alt_base in "ACGT":
                if alt_base == ref_codon[pos]:
                    continue

                alt_codon = (
                    ref_codon[:pos]
                    + alt_base
                    + ref_codon[pos + 1:]
                )

                if CODON_TO_AA[alt_codon] == alt_aa:
                    substitution = normalize_substitution(
                        ref_codon[pos], alt_base
                    )
                    counts[substitution] += 1

    total = sum(counts.values())

    if total == 0:
        return None

    return {
        key: value / total
        for key, value in counts.items()
    }


# 계산 규칙 검증
print("V600E:", infer_substitution_weights("V", "E"))
print("R175H:", infer_substitution_weights("R", "H"))


V600E: {'C>A': 0.0, 'C>G': 0.0, 'C>T': 0.0, 'T>A': 1.0, 'T>C': 0.0, 'T>G': 0.0}
R175H: {'C>A': 0.0, 'C>G': 0.0, 'C>T': 1.0, 'T>A': 0.0, 'T>C': 0.0, 'T>G': 0.0}


In [3]:
# mutation_signature(X) 구현

from functools import lru_cache

# V600E, R175H, Q61* 등만 허용
MUTATION_PATTERN = re.compile(r"^([A-Z])(\d+)([A-Z*])$")


@lru_cache(maxsize=None)
def get_mutation_weights(mutation):
    """변이 문자열을 6종 염기 치환 가중치로 변환."""
    match = MUTATION_PATTERN.fullmatch(mutation)

    if match is None:
        return None

    ref_aa, _, alt_aa = match.groups()

    # 동의 변이 제외
    if ref_aa == alt_aa:
        return None

    weights = infer_substitution_weights(ref_aa, alt_aa)

    if weights is None:
        return None

    return tuple(weights[key] for key in SIGNATURE_TYPES)


def mutation_signature(X):
    """
    환자별 아미노산 변이에서 6종 염기 치환 서명을 추정한다.

    Parameters
    ----------
    X : pd.DataFrame
        유전자별 변이 문자열이 담긴 데이터프레임

    Returns
    -------
    pd.DataFrame
        log1p 치환 개수 6열 + 치환 비율 6열
        + log1p 전체 개수 1열 = 총 13열 (float32)
    """
    counts = np.zeros((len(X), 6), dtype=np.float64)

    for row_idx, row in enumerate(X.itertuples(index=False, name=None)):
        for cell in row:
            if pd.isna(cell) or cell == "WT":
                continue

            for mutation in str(cell).split():
                weights = get_mutation_weights(mutation)

                if weights is not None:
                    counts[row_idx] += weights

    totals = counts.sum(axis=1)

    ratios = np.divide(
        counts,
        totals[:, None],
        out=np.zeros_like(counts),
        where=totals[:, None] != 0
    )

    features = np.column_stack([
        np.log1p(counts),
        ratios,
        np.log1p(totals)
    ]).astype(np.float32)

    columns = (
        [f"sig_log_{s.replace('>', '_')}" for s in SIGNATURE_TYPES]
        + [f"sig_ratio_{s.replace('>', '_')}" for s in SIGNATURE_TYPES]
        + ["sig_log_total"]
    )

    return pd.DataFrame(
        features,
        index=X.index,
        columns=columns
    )


print("함수 정의 완료")

함수 정의 완료


In [4]:
# 테스트 데이터 생성
X_sample = pd.DataFrame({
    "BRAF": ["V600E", "WT", "V600E", "WT"],
    "TP53": ["WT", "R175H", "R175H", "WT"]
})

# 변이 서명 피처 생성
result = mutation_signature(X_sample)

display(result.round(3))

# 결과 자동 검증
expected_counts = np.array([
    [0, 0, 0, 1, 0, 0],  # V600E
    [0, 0, 1, 0, 0, 0],  # R175H
    [0, 0, 1, 1, 0, 0],  # 둘 다
    [0, 0, 0, 0, 0, 0]   # 둘 다 WT
], dtype=np.float64)

expected_totals = expected_counts.sum(axis=1)

expected_ratios = np.divide(
    expected_counts,
    expected_totals[:, None],
    out=np.zeros_like(expected_counts),
    where=expected_totals[:, None] != 0
)

expected = np.column_stack([
    np.log1p(expected_counts),
    expected_ratios,
    np.log1p(expected_totals)
]).astype(np.float32)

np.testing.assert_allclose(
    result.to_numpy(),
    expected,
    rtol=1e-6,
    atol=1e-6
)

assert result.shape == (4, 13)
assert all(result.dtypes == np.float32)
assert result.index.equals(X_sample.index)

print("모든 테스트 통과!")

,sig_log_C_A,sig_log_C_G,sig_log_C_T,sig_log_T_A,sig_log_T_C,sig_log_T_G,sig_ratio_C_A,sig_ratio_C_G,sig_ratio_C_T,sig_ratio_T_A,sig_ratio_T_C,sig_ratio_T_G,sig_log_total
0,0.0,0.0,0.000,0.693,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.693
1,0.0,0.0,0.693,0.000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.693
2,0.0,0.0,0.693,0.693,0.0,0.0,0.0,0.0,0.5,0.5,0.0,0.0,1.099
3,0.0,0.0,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000


모든 테스트 통과!


In [5]:
# 실제 Train/Test에 변이 서명 피처 생성

# 1. 데이터 불러오기
train = pd.read_csv(PROJECT_ROOT / "data" / "train.csv")
test = pd.read_csv(PROJECT_ROOT / "data" / "test.csv")

X_train = train.drop(columns=["ID", "SUBCLASS"])
X_test = test.drop(columns=["ID"])

# 2. 변이 서명 피처 생성
train_sig = mutation_signature(X_train)
test_sig = mutation_signature(X_test)

# 3. 기본 검증
print("Train:", train_sig.shape)
print("Test:", test_sig.shape)

print("\n결측치 수")
print("Train:", train_sig.isna().sum().sum())
print("Test:", test_sig.isna().sum().sum())

print("\n무한대 값 수")
print("Train:", np.isinf(train_sig.to_numpy()).sum())
print("Test:", np.isinf(test_sig.to_numpy()).sum())

print("\n자료형:", train_sig.dtypes.unique())

assert train_sig.shape == (len(train), 13)
assert test_sig.shape == (len(test), 13)
assert train_sig.index.equals(X_train.index)
assert test_sig.index.equals(X_test.index)
assert all(train_sig.dtypes == np.float32)
assert all(test_sig.dtypes == np.float32)
assert np.isfinite(train_sig.to_numpy()).all()
assert np.isfinite(test_sig.to_numpy()).all()

print("\n모든 기본 검증 통과!")

display(train_sig.head())

Train: (6201, 13)
Test: (2546, 13)

결측치 수
Train: 0
Test: 0

무한대 값 수
Train: 0
Test: 0

자료형: [dtype('float32')]

모든 기본 검증 통과!


,sig_log_C_A,sig_log_C_G,sig_log_C_T,sig_log_T_A,sig_log_T_C,sig_log_T_G,sig_ratio_C_A,sig_ratio_C_G,sig_ratio_C_T,sig_ratio_T_A,sig_ratio_T_C,sig_ratio_T_G,sig_log_total
0,1.321756,1.909543,1.386294,0.348307,1.299283,1.485385,0.152778,0.319444,0.166667,0.023148,0.148148,0.189815,2.944439
1,0.847298,0.287682,1.992430,0.693147,0.000000,0.000000,0.148148,0.037037,0.703704,0.111111,0.000000,0.000000,2.302585
2,1.126011,1.276294,4.192177,2.025953,1.791759,1.522426,0.024510,0.030392,0.766667,0.077451,0.058824,0.042157,4.454347
3,0.916291,0.000000,0.000000,0.405465,1.098612,0.000000,0.375000,0.000000,0.000000,0.125000,0.500000,0.000000,1.609438
4,1.011601,1.126011,2.302585,0.882389,1.466337,1.858899,0.076087,0.090580,0.391304,0.061594,0.144928,0.235507,3.178054


In [6]:
# 변이 서명 피처의 분포 확인

# 6종 치환 유형별 비율 컬럼
ratio_cols = [
    f"sig_ratio_{s.replace('>', '_')}"
    for s in SIGNATURE_TYPES
]

# 변이 서명 생성 여부
train_valid = train_sig["sig_log_total"] > 0
test_valid = test_sig["sig_log_total"] > 0

print("=== 변이 서명 생성 현황 ===")
print(
    f"Train: {train_valid.sum():,}/{len(train_sig):,}명 "
    f"({train_valid.mean():.2%})"
)
print(
    f"Test: {test_valid.sum():,}/{len(test_sig):,}명 "
    f"({test_valid.mean():.2%})"
)

print("\n=== 변이 서명이 없는 샘플 ===")
print(f"Train: {(~train_valid).sum():,}명")
print(f"Test: {(~test_valid).sum():,}명")

# 유효한 샘플의 평균 치환 비율 비교
distribution = pd.DataFrame({
    "Train": train_sig.loc[train_valid, ratio_cols].mean(),
    "Test": test_sig.loc[test_valid, ratio_cols].mean()
})

distribution.index = SIGNATURE_TYPES

print("\n=== 염기 치환 6종 평균 비율 ===")
display(distribution.round(4))

# 비율 합계 검증
assert np.allclose(
    train_sig.loc[train_valid, ratio_cols].sum(axis=1),
    1.0,
    atol=1e-6
)
assert np.allclose(
    test_sig.loc[test_valid, ratio_cols].sum(axis=1),
    1.0,
    atol=1e-6
)

print("\n비율 합계 검증 통과!")

=== 변이 서명 생성 현황 ===
Train: 6,041/6,201명 (97.42%)
Test: 2,499/2,546명 (98.15%)

=== 변이 서명이 없는 샘플 ===
Train: 160명
Test: 47명

=== 염기 치환 6종 평균 비율 ===


,Train,Test
C>A,0.1555,0.1253
C>G,0.1191,0.1134
C>T,0.4094,0.3965
T>A,0.1123,0.1057
T>C,0.1348,0.1233
T>G,0.0688,0.1358



비율 합계 검증 통과!


In [ ]:
# 마지막 검증

X_edge = pd.DataFrame(
    {
        "BRAF": [
            "V600E V600E",  # 공백으로 분리된 복수 변이
            "V600V",        # 동의 변이
            "INVALID",      # 정규식 미일치
            np.nan,         # 결측치
            "V600E"         # 정상 변이
        ],
        "TP53": [
            "WT",
            "WT",
            "WT",
            "WT",
            "R175H"
        ]
    },
    index=[10, 20, 30, 40, 50]
)

edge_result = mutation_signature(X_edge)

# 복수 변이: T>A 2개
assert np.isclose(
    edge_result.loc[10, "sig_log_total"],
    np.log1p(2)
)

# 제외 대상: 13개 피처 모두 0
assert (edge_result.loc[[20, 30, 40]] == 0).all().all()

# 서로 다른 두 변이: 비율 각각 0.5
assert np.isclose(edge_result.loc[50, "sig_ratio_C_T"], 0.5)
assert np.isclose(edge_result.loc[50, "sig_ratio_T_A"], 0.5)

# 입력 인덱스 유지
assert edge_result.index.equals(X_edge.index)

print("추가 예외 케이스 검증 통과!")

추가 예외 케이스 검증 통과!
